# Synthetic Dataset Generation for PPRL

## Objective
To generate realistic synthetic datasets with controlled errors for evaluating PPRL methods.

## Why this matters
- Real data lacks ground truth
- Synthetic data allows controlled evaluation of linkage quality

### Bootstrap Cell

In [ ]:
# =================== BOOTSTRAP CELL ===================
# Standard setup for all notebooks
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0]  # assumes notebooks are in a subfolder
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ========================================================
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# from src.config.variables import COVARIATES

# ========================================================
# Optional for warnings and nicer plots
import warnings
warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

import sys
from pathlib import Path

# ========================================================
# 1️⃣ Ensure project root is in Python path
# Adjust this if your notebooks are nested deeper
PROJECT_ROOT = Path.cwd().parents[0]  # assumes notebooks are in a subfolder
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ========================================================
# 2️⃣ Import helper to load paths
from src.utils.helpers import load_paths

# ========================================================
# 3️⃣ Load paths from config.yaml (works regardless of notebook location)
paths = load_paths()

# ========================================================
# 4️⃣ Optionally, print paths to confirm
for key, value in paths.items():
    print(f"{key}: {value}")

# ========================================================
# 5️⃣ Now you can use these paths in your notebook:
# Example:
SYNTHETIC_DATA_DIR = paths['SYNTHETIC_DATA_DIR']
# OUT_DIR = paths['OUT_DIR']
# FIG_DIR = paths['FIG_DIR']

# ========================================================

### Imports

In [ ]:
import pandas as pd
import numpy as np
import random

### Creating a Base Population

In [ ]:
first_names = ["john", "mary", "jane", "peter", "faith", "mwangi", "otieno", "aisha"]
last_names = ["doe", "smith", "kamau", "otieno", "abdul", "okello", "njoroge"]

def generate_base_dataset(n=1000):
    data = []

    for i in range(n):
        record = {
            "id": i,
            "first_name": random.choice(first_names),
            "last_name": random.choice(last_names),
            "dob": str(random.randint(1960, 2010)) + "-" +
                   str(random.randint(1, 12)).zfill(2) + "-" +
                   str(random.randint(1, 28)).zfill(2)
        }
        data.append(record)

    return pd.DataFrame(data)

df_A = generate_base_dataset(1000)
# df_A.head()

In [ ]:
# df_A.tail()

### Introducing Realistic Errors

#### Typo generator

In [ ]:
def introduce_typo(text):
    if len(text) < 2:
        return text

    i = random.randint(0, len(text) - 2)
    return text[:i] + text[i+1] + text[i] + text[i+2:]

#### Missing values

In [ ]:
def introduce_missing(text, prob=0.1):
    return "" if random.random() < prob else text

#### Phonetic variation (simple)

In [ ]:
def phonetic_variation(name):
    replacements = {
        "ph": "f",
        "ck": "k",
        "ie": "y"
    }

    for k, v in replacements.items():
        name = name.replace(k, v)

    return name

### Dataset B (Corrupted Version)

In [ ]:
def create_corrupted_dataset(df, typo_prob=0.3, missing_prob=0.1):

    corrupted = []

    for _, row in df.iterrows():

        new_row = row.copy()

        # Introduce errors
        if random.random() < typo_prob:
            new_row["first_name"] = introduce_typo(new_row["first_name"])

        if random.random() < typo_prob:
            new_row["last_name"] = introduce_typo(new_row["last_name"])

        new_row["first_name"] = introduce_missing(new_row["first_name"], missing_prob)
        new_row["last_name"] = introduce_missing(new_row["last_name"], missing_prob)

        # Phonetic variation
        if random.random() < 0.2:
            new_row["first_name"] = phonetic_variation(new_row["first_name"])

        corrupted.append(new_row)

    return pd.DataFrame(corrupted)

df_B = create_corrupted_dataset(df_A)
# df_B.head()

### True Match Labels

In [ ]:
true_matches = df_A[["id"]].copy()
true_matches["id_B"] = true_matches["id"]

true_matches.columns = ["id_A", "id_B"]
true_matches.head()

### Non-Matching Records

In [ ]:
def add_non_matches(df, n_extra=200):

    extra = generate_base_dataset(n_extra)
    extra["id"] = range(df["id"].max() + 1, df["id"].max() + 1 + n_extra)

    return pd.concat([df, extra], ignore_index=True)

df_B = add_non_matches(df_B)

### Save Datasets

In [ ]:
df_A.to_csv(SYNTHETIC_DATA_DIR / "dataset_A.csv", index=False)
df_B.to_csv(SYNTHETIC_DATA_DIR / "dataset_B.csv", index=False)
true_matches.to_csv(SYNTHETIC_DATA_DIR / "true_matches.csv", index=False)
